# Part 2 — External Database Configuration

*Last updated:* 2026-01-08

This notebook is the step-by-step guide for creating and validating the **external YAML** files that control which **external databases**
(HGNC, MGI, UniProt, RefSeq, …) are included when IDTrack builds the identifier graph.

By the end, you will:
- have `<organism>_externals_modified.yml` files in your local repository
- understand why keeping your external database selection **small and curated** improves mapping quality
- know how to validate that your YAML is compatible with your chosen snapshot boundary

This notebook covers:
- **2.1 Human** (*Homo sapiens*) — multi-assembly (GRCh38 + GRCh37, and older archives when available)
- **2.2 Mouse** (*Mus musculus*) — clean handoff (one maintained assembly per release: GRCm37 → GRCm38 → GRCm39)
- **2.3 Pig** (*Sus scrofa*) — clean handoff (one maintained assembly per release: Sscrofa9.2 → Sscrofa10.2 → Sscrofa11.1)
- **2.4 Adding a new organism** (advanced; may require a small code configuration step)

In IDTrack, assemblies are a first-class dimension (not just “primary vs legacy”): the templates keep all assembly entries that Ensembl
exposes for the species. Keeping them is usually the right choice, especially when you integrate datasets annotated with different GTFs or
reference packages.

> **Tip:** If you're new, start with `00_idtrack_overview.ipynb` (concepts) and `01_installation_guide.ipynb` (setup).


## 2.0 — Pre-requisites (what you need before you start)

- A working Python environment with IDTrack installed (`pip install idtrack`)
- Network access (first-time runs download Ensembl metadata)
- A writable **local repository** folder (IDTrack cache)

**What you get at the end:**
- `homo_sapiens_externals_modified.yml`
- `mus_musculus_externals_modified.yml`
- `sus_scrofa_externals_modified.yml`

Each file lives in your local repository and is safe to share with collaborators.


In [1]:
# Load notebook utilities (collapsible output magic for tutorials)
%load_ext _notebook_utils

In [2]:
# 1) Setup (run this once)
from __future__ import annotations

import os
from pathlib import Path

import yaml
import idtrack

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', './idtrack_cache')).resolve()
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api.configure_logger()

print('Local repository:', LOCAL_REPOSITORY)

Local repository: /Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache


### What is this local repository?

Think of it as your **IDTrack workspace**. It will contain:
- cached Ensembl tables (downloaded once, reused many times)
- graph snapshot files (`graph_<organism>_... .pickle`)
- your external YAML files (`*_externals_modified.yml`)

If you work on multiple projects, you can use one shared local repository (big but convenient),
or separate local repositories (clean separation, easier to archive/share).


## 2. External YAML in plain language

The YAML answers the question:

> **Which external databases should IDTrack trust and include as edges in the graph?**

Ensembl knows about *many* external resources. Some are high quality and useful, some are redundant, and some
create a lot of branching (ambiguity).

A good rule of thumb:
- enable a **small, curated** set of strong databases
- avoid enabling lots of near-duplicates

### YAML structure (what you will see)

The generated template is nested like this:

```yaml
<organism>:
  gene:
    <Database Name>:
      Assembly:
        <assembly_code>:
          Ensembl release: "90,91,92,..."
          Include: false
      Database Index: 123
      Potential Synonymous: ""
```

**You mainly edit one thing:** `Include: false` → `Include: true`.

Why is there an `Assembly:` level?

- Some external databases are only available (or only well-populated) on specific assembly/release combinations.
- IDTrack can map across assemblies, so it needs to know which edges exist in which build context.

For most curated databases, enabling them for **all assemblies listed in the template** is the right default.


## 3. Choose your snapshot release (reproducibility knob)

When you build graphs later, you will pick a **snapshot release** (maximum Ensembl release).

For beginners, the best choice is usually:
- **the latest release** for the organism

For reproducible research projects, consider pinning:
- the Ensembl release used by your reference annotation (e.g. the one your pipeline used)
- or the release used by a published dataset you integrate


## 2.1 — Human (Homo sapiens)

Human is special in one convenient way: IDTrack ships a **default external YAML** for human.

An important conceptual point before you start editing:

- Assemblies are not just “primary vs legacy” in IDTrack.
- Assemblies are part of the mapping problem.

This matters in real projects (especially atlas building), where you often combine datasets annotated with different genome builds:
- GRCh37-era references / GTFs
- GRCh38-era references / GTFs

IDTrack’s snapshot graphs are **multi-assembly**. Keeping multiple assembly blocks enabled in the external YAML helps the path-finder:
- interpret input identifiers in the correct assembly context
- map across assemblies when needed (in addition to mapping across releases)
- use external databases that exist only on specific assemblies as additional “bridges”

For human, the shipped default YAML already includes the common human assembly codes exposed by Ensembl (typically `38` = GRCh38 and
`37` = GRCh37). Depending on release coverage, you may also see older archive assemblies in templates.

You have two good options:

1. **Use the shipped default as a starting point** (fast, recommended for most users)
2. **Regenerate a fresh template from live Ensembl metadata** (slower, but useful if you want to refresh external database lists)

Recommended external databases for human (high signal, widely used):
- **HGNC Symbol** (human-readable gene symbols)
- **EntrezGene**
- **UniProtKB**
- **RefSeq_mRNA** (optionally also RefSeq proteins if your workflow needs them)

> **Warning:** Assembly codes are species-specific. `38` means GRCh38 for human, but `38` means GRCm38 for mouse.

> **Tip:** Keep your allowlist small at first. You can expand later once you understand how ambiguity shows up in your results.


In [3]:
# 4.1 Resolve organism name + latest release
organism_hs, latest_release_hs = api.resolve_organism('human')
SNAPSHOT_RELEASE_HS = latest_release_hs  # change if you want to pin
organism_hs, SNAPSHOT_RELEASE_HS


2026-01-11 14:46:55 INFO:verify_organism: Ensembl Rest API query to get the organism names and associated releases.


('homo_sapiens', 115)

### 2.1.1 Create a DatabaseManager snapshot

This object is responsible for talking to Ensembl (downloads + caching) **bounded by your snapshot release**.


In [4]:
dm_hs = api.get_database_manager(organism_name=organism_hs, snapshot_release=SNAPSHOT_RELEASE_HS)
dm_hs


2026-01-11 14:47:22 INFO:database_manager: Using assembly-specific release range for homo_sapiens assembly 38: releases 76-115 (from config [76, None])


### 2.1.2 Generate a template YAML (optional for human, required for mouse/pig)

This step can take time, because IDTrack enumerates database metadata across releases.
You typically do it once per organism (and then keep the YAML).


In [5]:
%%collapse Click to show download logs
# Included for tutorial purposes only.

# NOTE: This can take a while on first run (network + caching).
df_hs = dm_hs.create_database_content(just_download=False)
dm_hs.external_inst.create_template_yaml(df_hs)

template_hs = Path(dm_hs.external_inst.file_name_template_yaml())
modified_hs = Path(dm_hs.external_inst.file_name_modified_yaml(mode='configured'))
{
    'template_yaml': str(template_hs),
    'modified_yaml': str(modified_hs),
}

{'template_yaml': '/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/homo_sapiens_externals_template.yml',
 'modified_yaml': '/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/homo_sapiens_externals_modified.yml'}

{'template_yaml': '/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/homo_sapiens_externals_template.yml',
 'modified_yaml': '/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/homo_sapiens_externals_modified.yml'}

### 2.1.3 Edit the YAML (two options)

#### Option A — edit by hand (recommended at least once)
1. Open the template file shown above (ends with `_externals_template.yml`).
2. Search for a database you care about (e.g. `HGNC Symbol`).
3. Change `Include: false` to `Include: true` for **every assembly block listed** (recommended).
   - This keeps the database available as a bridge regardless of which genome build your inputs were annotated against.
   - Only restrict to a subset of assemblies if you intentionally want a build-specific configuration.
4. Save as `_externals_modified.yml` in the same folder (IDTrack will look for this first).

#### Option B — programmatic toggles (great for reproducibility)
The next cell applies the **shipped default configuration** used by IDTrack (`homo_sapiens_externals_modified.yml`).

This default enables **49 form/database/assembly combinations** across three forms:
- **gene** (28 combinations): HGNC Symbol, EntrezGene, NCBI gene, UniProtKB Gene Name, Havana gene, Vega gene, RFAM, Clone-based identifiers, and their synonym variants
- **transcript** (12 combinations): CCDS, Havana transcript, RefSeq mRNA/ncRNA (curated + predicted)
- **translation** (9 combinations): Havana translation, RefSeq peptide (curated + predicted), UniProt (Swiss-Prot + TrEMBL)

You can modify `DEFAULT_HS_EXTERNALS` in the cell below to customize your allowlist.

In [6]:
# Default external databases for HUMAN (matches the shipped default config)
# This dict is structured as: form -> database -> list of assemblies where Include=True
# These 49 combinations are the default used by IDTrack for homo_sapiens.

DEFAULT_HS_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [36, 37, 38],
        'Clone_based_vega_gene': [36, 37, 38],
        'EntrezGene': [36, 37, 38],
        'HGNC Symbol': [36, 37, 38],
        'Havana gene': [36, 37, 38],
        'NCBI gene': [36, 37, 38],
        'NCBI gene (formerly Entrezgene)': [36, 37, 38],
        'RFAM': [36, 37, 38],
        'UniProtKB Gene Name': [36, 37, 38],
        'Vega gene': [36, 37, 38],
        'Vega_gene': [36, 37, 38],
        'synonym_id::EntrezGene': [36, 37, 38],
        'synonym_id::HGNC Symbol': [36, 37, 38],
        'synonym_id::NCBI gene': [36, 37, 38],
        'synonym_id::NCBI gene (formerly Entrezgene)': [36, 37, 38],
        'synonym_id::UniProtKB Gene Name': [36, 37, 38],
    },
    'transcript': {
        'CCDS': [36, 37, 38],
        'Havana transcript': [36, 37, 38],
        'RefSeq_mRNA': [36, 37, 38],
        'RefSeq_mRNA_predicted': [36, 37, 38],
        'RefSeq_ncRNA': [36, 37, 38],
        'RefSeq_ncRNA_predicted': [36, 37, 38],
    },
    'translation': {
        'Havana translation': [36, 37, 38],
        'RefSeq_peptide': [36, 37, 38],
        'RefSeq_peptide_predicted': [36, 37, 38],
        'Uniprot/SPTREMBL': [36, 37, 38],
        'Uniprot/SWISSPROT': [36, 37, 38],
    },
}

# Load the template YAML
y = yaml.safe_load(template_hs.read_text(encoding='utf-8'))

# Apply the default configuration: set Include=True for matching form/database/assembly
enabled_count = 0
for form, databases in DEFAULT_HS_EXTERNALS.items():
    if form not in y[organism_hs]:
        continue
    for db_name, assemblies in databases.items():
        if db_name not in y[organism_hs][form]:
            print(f'  [SKIP] {form}/{db_name} not found in template')
            continue
        for asm_code in assemblies:
            asm_str = str(asm_code)
            if asm_str in y[organism_hs][form][db_name]['Assembly']:
                y[organism_hs][form][db_name]['Assembly'][asm_str]['Include'] = True
                enabled_count += 1

modified_hs.write_text(yaml.safe_dump(y, sort_keys=False, allow_unicode=True), encoding='utf-8')

print(f'Enabled {enabled_count} form/database/assembly combinations (the shipped default).')
print('Wrote:', modified_hs)

Enabled 63 form/database/assembly combinations (the shipped default).
Wrote: /Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/homo_sapiens_externals_modified.yml


### 2.1.4 Validate + preview your selections

This load step is important: it confirms your YAML contains your snapshot release and that IDTrack can parse it.


In [7]:
_ = dm_hs.external_inst.load_modified_yaml()
print('Enabled DBs:', dm_hs.external_inst.give_list_for_case('db'))
print('Assemblies in play:', dm_hs.external_inst.give_list_for_case('assembly'))

# Your dm_hs was created with snapshot_release=115, but GRCh36 data in Ensembl only goes up to 
# around release 54-55 (when it was superseded by GRCh37). 
_dm_hs = dm_hs.change_release_auto_assembly(50)
_ = _dm_hs.external_inst.load_modified_yaml()
print('Enabled DBs:', _dm_hs.external_inst.give_list_for_case('db'))
print('Assemblies in play:', _dm_hs.external_inst.give_list_for_case('assembly'))

Enabled DBs: ['NCBI gene (formerly Entrezgene)', 'UniProtKB Gene Name', 'synonym_id::NCBI gene (formerly Entrezgene)', 'RFAM', 'EntrezGene', 'synonym_id::HGNC Symbol', 'HGNC Symbol']
Assemblies in play: [37, 38]
Enabled DBs: ['Havana gene', 'Clone_based_vega_gene', 'Clone_based_ensembl_gene', 'synonym_id::HGNC Symbol', 'HGNC Symbol']
Assemblies in play: [36]


## 2.2 — Mouse (Mus musculus)

Mouse does not ship with a default external YAML in the package, so the usual workflow is:

1. generate a template YAML (from Ensembl metadata)
2. enable a curated allowlist of databases
3. validate the YAML


Mouse templates list multiple assemblies across history, but Ensembl is a clean-handoff species (one maintained assembly per release). You will usually see assembly codes like:
- `39` = GRCm39
- `38` = GRCm38
- `37` = GRCm37

> **Tip:** For clean-handoff species, enabling the same database across the listed assemblies is usually fine;
> the per-assembly release ranges are disjoint, so you typically do not get overlapping assemblies within a single release.

Mouse-specific, commonly useful databases:
- **MGI Symbol** (mouse gene symbols)
- **EntrezGene**
- **UniProtKB**
- **RefSeq_mRNA**

> **Expected output:** You will create `mus_musculus_externals_modified.yml` in your local repository.


In [8]:
# 5.1 Resolve organism name + latest release
organism_mm, latest_release_mm = api.resolve_organism('mus musculus')
SNAPSHOT_RELEASE_MM = latest_release_mm
organism_mm, SNAPSHOT_RELEASE_MM


2026-01-11 14:48:00 INFO:verify_organism: Ensembl Rest API query to get the organism names and associated releases.


('mus_musculus', 115)

In [9]:
# 5.2 Create DatabaseManager snapshot
dm_mm = api.get_database_manager(organism_name=organism_mm, snapshot_release=SNAPSHOT_RELEASE_MM)
dm_mm


2026-01-11 14:48:26 INFO:database_manager: Using assembly-specific release range for mus_musculus assembly 39: releases 103-115 (from config [103, None])


In [10]:
%%collapse Click to show download logs
# Included for tutorial purposes only.

# 5.3 Generate template YAML
df_mm = dm_mm.create_database_content(just_download=False)
dm_mm.external_inst.create_template_yaml(df_mm)

template_mm = Path(dm_mm.external_inst.file_name_template_yaml())
modified_mm = Path(dm_mm.external_inst.file_name_modified_yaml(mode='configured'))
{
    'template_yaml': str(template_mm),
    'modified_yaml': str(modified_mm),
}

{'template_yaml': '/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/mus_musculus_externals_template.yml',
 'modified_yaml': '/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/mus_musculus_externals_modified.yml'}

{'template_yaml': '/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/mus_musculus_externals_template.yml',
 'modified_yaml': '/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/mus_musculus_externals_modified.yml'}

### 2.2.1 Programmatic allowlist (mouse)

Mouse gene symbols come from **MGI** (Mouse Genome Informatics), so a typical set includes `MGI Symbol`.


In [11]:
# Default external databases for MOUSE.
# This dict is structured as: form -> database -> list of assemblies where Include=True

DEFAULT_MM_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [37, 38, 39],
        'Clone_based_vega_gene': [37, 38, 39],
        'EntrezGene': [37, 38, 39],
        'MGI Symbol': [37, 38, 39],
        'Havana gene': [37, 38, 39],
        'NCBI gene': [37, 38, 39],
        'NCBI gene (formerly Entrezgene)': [37, 38, 39],
        'RFAM': [37, 38, 39],
        'UniProtKB Gene Name': [37, 38, 39],
        'Vega gene': [37, 38, 39],
        'Vega_gene': [37, 38, 39],
        'synonym_id::EntrezGene': [37, 38, 39],
        'synonym_id::MGI Symbol': [37, 38, 39],
        'synonym_id::NCBI gene': [37, 38, 39],
        'synonym_id::NCBI gene (formerly Entrezgene)': [37, 38, 39],
        'synonym_id::UniProtKB Gene Name': [37, 38, 39],
    },
    'transcript': {
        'CCDS': [37, 38, 39],
        'Havana transcript': [37, 38, 39],
        'RefSeq_mRNA': [37, 38, 39],
        'RefSeq_mRNA_predicted': [37, 38, 39],
        'RefSeq_ncRNA': [37, 38, 39],
        'RefSeq_ncRNA_predicted': [37, 38, 39],
    },
    'translation': {
        'Havana translation': [37, 38, 39],
        'RefSeq_peptide': [37, 38, 39],
        'RefSeq_peptide_predicted': [37, 38, 39],
        'Uniprot/SPTREMBL': [37, 38, 39],
        'Uniprot/SWISSPROT': [37, 38, 39],
    },
}

y = yaml.safe_load(template_mm.read_text(encoding='utf-8'))

# Apply the default configuration: set Include=True for matching form/database/assembly
enabled_count = 0
for form, databases in DEFAULT_MM_EXTERNALS.items():
    if form not in y[organism_mm]:
        continue
    for db_name, assemblies in databases.items():
        if db_name not in y[organism_mm][form]:
            print(f'  [SKIP] {form}/{db_name} not found in template')
            continue
        for asm_code in assemblies:
            asm_str = str(asm_code)
            if asm_str in y[organism_mm][form][db_name]['Assembly']:
                y[organism_mm][form][db_name]['Assembly'][asm_str]['Include'] = True
                enabled_count += 1

modified_mm.write_text(yaml.safe_dump(y, sort_keys=False, allow_unicode=True), encoding='utf-8')

print(f'Enabled {enabled_count} form/database/assembly combinations.')
print('Wrote:', modified_mm)


Enabled 67 form/database/assembly combinations.
Wrote: /Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/mus_musculus_externals_modified.yml


In [12]:
_ = dm_mm.external_inst.load_modified_yaml()
print('Enabled DBs:', sorted(dm_mm.external_inst.give_list_for_case('db')))
print('Assemblies in play:', sorted(dm_mm.external_inst.give_list_for_case('assembly')))


Enabled DBs: ['EntrezGene', 'MGI Symbol', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'UniProtKB Gene Name', 'synonym_id::MGI Symbol', 'synonym_id::NCBI gene (formerly Entrezgene)']
Assemblies in play: [39]


## 2.3 — Pig (Sus scrofa)

Pig templates list multiple assemblies across history, but Ensembl is a clean-handoff species (one maintained assembly per release). Common Ensembl assembly codes you will see include:
- `111` = Sscrofa11.1
- `102` = Sscrofa10.2
- `9` = Sscrofa9.2

> **Tip:** For clean-handoff species, enabling the same database across the listed assemblies is usually fine;
> older assemblies mainly matter for legacy datasets and archive releases.



Pig also requires generating a template YAML and then enabling a curated allowlist.

Pig datasets often mix different naming conventions, so enabling a small set of high-signal externals is especially important.

Commonly useful databases for pig:
- **EntrezGene**
- **UniProtKB**
- **RefSeq_mRNA**

> **Expected output:** You will create `sus_scrofa_externals_modified.yml` in your local repository.


In [13]:
# 6.1 Resolve organism name + latest release
organism_ss, latest_release_ss = api.resolve_organism('sus scrofa')
SNAPSHOT_RELEASE_SS = latest_release_ss
organism_ss, SNAPSHOT_RELEASE_SS


2026-01-11 18:01:57 INFO:verify_organism: Ensembl Rest API query to get the organism names and associated releases.


('sus_scrofa', 115)

In [14]:
# 6.2 Create DatabaseManager snapshot
dm_ss = api.get_database_manager(organism_name=organism_ss, snapshot_release=SNAPSHOT_RELEASE_SS)
dm_ss


2026-01-11 18:02:27 INFO:database_manager: Using assembly-specific release range for sus_scrofa assembly 111: releases 90-115 (from config [90, None])


In [15]:
%%collapse Click to show download logs
# 6.3 Generate template YAML
df_ss = dm_ss.create_database_content(just_download=False)
dm_ss.external_inst.create_template_yaml(df_ss)

template_ss = Path(dm_ss.external_inst.file_name_template_yaml())
modified_ss = Path(dm_ss.external_inst.file_name_modified_yaml(mode='configured'))
{
    'template_yaml': str(template_ss),
    'modified_yaml': str(modified_ss),
}

{'template_yaml': '/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/sus_scrofa_externals_template.yml',
 'modified_yaml': '/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/sus_scrofa_externals_modified.yml'}

{'template_yaml': '/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/sus_scrofa_externals_template.yml',
 'modified_yaml': '/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/sus_scrofa_externals_modified.yml'}

### 2.3.1 Programmatic allowlist (pig)

Pig symbol databases can vary across releases. A safe starter set often includes `EntrezGene` and `UniProtKB`.
Use the template to inspect what is available for your snapshot release.


In [16]:
# Default external databases for PIG.
# This dict is structured as: form -> database -> list of assemblies where Include=True

DEFAULT_SS_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [9, 102, 111],
        'Clone_based_vega_gene': [9, 102, 111],
        'EntrezGene': [9, 102, 111],
        'HGNC Symbol': [9, 102, 111],
        'VGNC Symbol': [9, 102, 111],
        'Havana gene': [9, 102, 111],
        'NCBI gene': [9, 102, 111],
        'NCBI gene (formerly Entrezgene)': [9, 102, 111],
        'RFAM': [9, 102, 111],
        'UniProtKB Gene Name': [9, 102, 111],
        'synonym_id::EntrezGene': [9, 102, 111],
        'synonym_id::HGNC Symbol': [9, 102, 111],
        'synonym_id::NCBI gene': [9, 102, 111],
        'synonym_id::NCBI gene (formerly Entrezgene)': [9, 102, 111],
        'synonym_id::UniProtKB Gene Name': [9, 102, 111],
    },
    'transcript': {
        'Havana transcript': [9, 102, 111],
        'RefSeq_mRNA': [9, 102, 111],
        'RefSeq_mRNA_predicted': [9, 102, 111],
        'RefSeq_ncRNA': [9, 102, 111],
        'RefSeq_ncRNA_predicted': [9, 102, 111],
    },
    'translation': {
        'RefSeq_peptide': [9, 102, 111],
        'RefSeq_peptide_predicted': [9, 102, 111],
        'Uniprot/SPTREMBL': [9, 102, 111],
        'Uniprot/SWISSPROT': [9, 102, 111],
    },
}

y = yaml.safe_load(template_ss.read_text(encoding='utf-8'))

# Apply the default configuration: set Include=True for matching form/database/assembly
enabled_count = 0
for form, databases in DEFAULT_SS_EXTERNALS.items():
    if form not in y[organism_ss]:
        continue
    for db_name, assemblies in databases.items():
        if db_name not in y[organism_ss][form]:
            print(f'  [SKIP] {form}/{db_name} not found in template')
            continue
        for asm_code in assemblies:
            asm_str = str(asm_code)
            if asm_str in y[organism_ss][form][db_name]['Assembly']:
                y[organism_ss][form][db_name]['Assembly'][asm_str]['Include'] = True
                enabled_count += 1

modified_ss.write_text(yaml.safe_dump(y, sort_keys=False, allow_unicode=True), encoding='utf-8')

print(f'Enabled {enabled_count} form/database/assembly combinations.')
print('Wrote:', modified_ss)


Enabled 54 form/database/assembly combinations.
Wrote: /Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/sus_scrofa_externals_modified.yml


In [17]:
_ = dm_ss.external_inst.load_modified_yaml()
print('Enabled DBs:', sorted(dm_ss.external_inst.give_list_for_case('db')))
print('Assemblies in play:', sorted(dm_ss.external_inst.give_list_for_case('assembly')))


Enabled DBs: ['EntrezGene', 'HGNC Symbol', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'UniProtKB Gene Name', 'VGNC Symbol', 'synonym_id::NCBI gene (formerly Entrezgene)']
Assemblies in play: [111]


## 2.4 — Adding a New Organism (Advanced)

IDTrack currently ships with built-in support for a small set of organisms (human/mouse/pig). If you want to use a different
Ensembl-supported species, there are **two layers** to set up:

1. **Core configuration (required):** IDTrack must know the canonical Ensembl species name and which numeric **assembly codes**
   appear in schema names like `<organism>_core_<release>_<assembly>` (configure `idtrack/_db.py`).
   - Direct MySQL connectivity is optional: when ports are blocked, IDTrack downloads the same tables from the HTTPS/FTP MySQL dumps.
2. **External YAML (required for externals):** once the organism/assemblies are configured, you generate a template YAML and curate an
   allowlist exactly like mouse/pig above.

Practical recipe:

- Step A: Resolve the canonical Ensembl species name (snake_case) with `api.resolve_organism(...)`.
- Step B: Configure the organism and its relevant assemblies in `idtrack/_db.py` by extending `DB.assembly_mysqlport_priority`.
  - Listing multiple assemblies is what enables cross-assembly mapping and assembly-scoped external databases.
- Step C: Re-run the YAML-template generation workflow (`DatabaseManager.create_database_content(...)` → `create_template_yaml(...)`).
- Step D: Build a graph snapshot (Part 3) and run the sanity checks.

> **Warning:** Until Step B is done, `DatabaseManager` will raise `NotImplementedError` for that organism.


In [18]:
# Minimal helper cell: check if an organism is supported by *this* IDTrack version.
# (Safe to run; it will not modify your installation.)

from idtrack._db import DB

print('Supported organisms in this IDTrack version:')
print('  ' + ', '.join(DB.supported_organisms))

# Example: resolve an organism name via Ensembl REST (works even if the organism isn't configured locally)
organism_query = 'danio rerio'  # zebrafish (example)
formal_name, latest_release = api.resolve_organism(organism_query)
print('Resolved via Ensembl REST ->', organism_query, '→', formal_name, '(latest release:', latest_release, ')')

if formal_name not in DB.supported_organisms:
    print()
    print('This organism is not yet configured in this IDTrack version.')
    print('To add it: extend DB.assembly_mysqlport_priority in idtrack/_db.py with its assembly code + port list,')
    print('then regenerate the external YAML using the same workflow as mouse/pig above.')
else:
    print()
    print('This organism is already configured. You can now generate a template YAML and continue.')


2026-01-11 19:15:21 INFO:verify_organism: Ensembl Rest API query to get the organism names and associated releases.


Supported organisms in this IDTrack version:
  homo_sapiens, mus_musculus, sus_scrofa
Resolved via Ensembl REST -> danio rerio → danio_rerio (latest release: 115 )

This organism is not yet configured in this IDTrack version.
To add it: extend DB.assembly_mysqlport_priority in idtrack/_db.py with its assembly code + port list,
then regenerate the external YAML using the same workflow as mouse/pig above.


## 2.5 — Final checklist (before you build graphs)

You should now have these three files in your local repository:
- `homo_sapiens_externals_modified.yml`
- `mus_musculus_externals_modified.yml`
- `sus_scrofa_externals_modified.yml`

The next notebook (`03_initialization_graph.ipynb`) will build graphs using these configs.


In [19]:
# Quick existence check
expected = [
    LOCAL_REPOSITORY / 'homo_sapiens_externals_modified.yml',
    LOCAL_REPOSITORY / 'mus_musculus_externals_modified.yml',
    LOCAL_REPOSITORY / 'sus_scrofa_externals_modified.yml',
]
for p in expected:
    print('OK' if p.exists() else 'MISSING', '-', p)


OK - /Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/homo_sapiens_externals_modified.yml
OK - /Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/mus_musculus_externals_modified.yml
OK - /Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/sus_scrofa_externals_modified.yml


## 2.6 — Best practices (once you get comfortable)

1. **Keep allowlists small**: enabling many overlapping databases often increases ambiguity.
2. **Multiple profiles**: you can maintain different YAMLs per project (e.g. ‘strict’ vs ‘broad’).
3. **Assembly awareness**:
   - In Ensembl schema names, the last number encodes the genome assembly (e.g. human 38 = GRCh38; mouse 39 = GRCm39; pig 111 = Sscrofa11.1).
   - Assembly codes are species-specific, and external databases can be assembly-scoped.
   - Keeping all assembly blocks in your YAML is usually beneficial when you integrate mixed-build datasets.
4. **Re-running**: you can regenerate templates after a new Ensembl release and re-apply your allowlist.
